In [1]:
#Imports and Client

from openai import OpenAI
import os
from dotenv import load_dotenv
import json

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)


In [2]:
#Contact extraction schema
contact_schema = {
    "type": "object",
    "properties": {
        "name": {"type": ["string", "null"]},
        "email": {"type": ["string", "null"]},
        "company": {"type": ["string", "null"]}
    },
    "required": ["name", "email", "company"],
    "additionalProperties": False
}

In [3]:
#Contact extraction function
def extract_contact(text):

    response = client.responses.create(
        model="gpt-4.1-mini",
        instructions="""
        Extract the name, email, and company.
        If information is missing, return null.
        Never invent information.
        """,
        input=text,
        text={
            "format": {
                "type": "json_schema",
                "name": "contact_extraction",
                "strict": True,
                "schema": contact_schema
            }
        }
    )

    return json.loads(response.output_text)

In [4]:
#Callable tool
def count_words(text):
    return len(text.split())

In [5]:
#Describing the tool to the model

tools = [
    {
        "type": "function",
        "name": "count_words",
        "description": "Count the words in the provided text.",
        "parameters": {
            "type": "object",
            "properties": {
                "text": {
                    "type": "string"
                }
            },
            "required": ["text"],
            "additionalProperties": False
        },
        "strict": True
    }
]

In [6]:
def get_word_count(text):

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=f"Use the count_words tool to count the words in this text:\n{text}",
        tools=tools,
        tool_choice="required"
    )

    for item in response.output:

        if item.type == "function_call":

            if item.name != "count_words":
                raise ValueError("Unexpected tool requested")

            arguments = json.loads(item.arguments)

            result = count_words(**arguments)

            tool_response = client.responses.create(
                model="gpt-4.1-mini",
                previous_response_id=response.id,
                input=[
                    {
                        "type": "function_call_output",
                        "call_id": item.call_id,
                        "output": str(result)
                    }
                ]
            )

            return result

    raise RuntimeError("The model did not request the expected tool.")

In [7]:
#Combining things
def process_text(text):

    contact = extract_contact(text)

    word_count = get_word_count(text)

    result = {
        "contact": contact,
        "word_count": word_count
    }

    return result

In [8]:
#Testing
sample_text = """
Hello!

My name is David Miller.
I work at Microsoft.
My email is david@microsoft.com.

Looking forward to connecting!
"""

result = process_text(sample_text)

print(json.dumps(result, indent=4))

{
    "contact": {
        "name": "David Miller",
        "email": "david@microsoft.com",
        "company": "Microsoft"
    },
    "word_count": 18
}
